In [ ]:
import pandas as pd
import re
from datasets import Dataset
from transformers import AutoTokenizer

print("1. Reading original data...")
df = pd.read_excel("dataset_indobert_fix.xlsx")

df['Label'] = pd.to_numeric(df['Label'], errors='coerce')
df = df.dropna(subset=['Teks', 'Label'])
df['Label'] = df['Label'].astype(int)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()
    return text

print("2. Cleaning text")
df['Clean_Text'] = df['Teks'].apply(clean_text)

print("3. Converting to AI format...")
df_clean = df[['Clean_Text', 'Label']]
dataset_hf = Dataset.from_pandas(df_clean)

print("4. Splitting data (80% Train, 20% Test)")
dataset_split = dataset_hf.train_test_split(test_size=0.2, seed=42)

print("5. Tokenization (Translating text to numbers for IndoBERT)")
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

def process_tokenization(example):
    return tokenizer(example["Clean_Text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = dataset_split["train"].map(process_tokenization, batched=True)
tokenized_test = dataset_split["test"].map(process_tokenization, batched=True)

print("6. Saving the prepared dataset to a new folder...")
tokenized_train.save_to_disk("dataset_asli_finetune/train")
tokenized_test.save_to_disk("dataset_asli_finetune/test")

print("Phase 1 is 100% Complete move on to File 2.")

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Reading original data...
2. Cleaning text...
3. Converting to AI format...
4. Splitting data (80% Train, 20% Test)...
5. Tokenization (Translating text to numbers for IndoBERT)...


Map: 100%|██████████| 576/576 [00:00<00:00, 10779.72 examples/s]


6. Saving the prepared dataset to a new folder...


Saving the dataset (1/1 shards): 100%|██████████| 576/576 [00:00<00:00, 64429.66 examples/s]

Phase 1 is 100% Complete! Move on to File 2.
